# 🔁 Laboratorio 2: Redes Neuronales Recurrentes (RNN, LSTM y GRU)
## Casos de Estudio: Pronóstico del Ciclo Solar y Memoria a Largo Plazo

**Pontificia Universidad Javeriana Cali — Curso de Deep Learning**  
**Tutor:** Jan Polanco Velasco  
**Semestre:** 2026-2

---

### 🎯 Objetivo del Laboratorio

En el **Laboratorio 1** pronosticamos las manchas solares mensuales con una red **MLP autorregresiva** que recibía una ventana *plana* de 24 retardos. En este laboratorio damos el salto a las **arquitecturas recurrentes**, capaces de procesar la secuencia **paso a paso** y mantener un **estado de memoria** interna.

Mantenemos el mismo dataset (*Monthly Sunspots*) para que la comparación con el Lab 1 sea directa. Además, entrenamos las tres arquitecturas sobre una **tarea de memoria a largo plazo** (*adding problem*) que evidencia de forma dramática el **desvanecimiento del gradiente** en la RNN simple.

### 📋 Contenido

1. Marco teórico: del vector plano al tensor tridimensional
2. Carga y preprocesamiento de la serie temporal
3. Formulación de secuencias: `[samples, time_steps, features]`
4. Modelo LSTM: arquitectura, entrenamiento y curvas de aprendizaje
5. *Benchmarking*: MLP vs. Simple RNN vs. LSTM vs. GRU
6. Predicción a un paso y análisis comparativo
7. Inferencia recurrente paso a paso y proyecciones futuras
8. Memoria a largo plazo: el talón de Aquiles de la RNN simple
9. Conclusiones

> 💡 **Consejo:** ejecuta las celdas en orden. Las gráficas de Plotly son interactivas (haz *zoom*, pasa el cursor y usa la leyenda).

---
## 1. 📚 Marco Teórico: de la Ventana Plana a la Secuencia

### 1.1 Revisitando el Laboratorio 1

En el Lab 1 el modelo recibía un **vector plano** de $n = 24$ retardos:

$$\underbrace{[T_{k-24},\, \dots,\, T_{k-1}]}_{X \in \mathbb{R}^{24}} \;\xrightarrow{\;f_\theta\;}\; T_k$$

Cada retardo ocupaba una posición **fija** con su propio peso. La red *no* tenía un mecanismo explícito de memoria: al cambiar la ventana, todo el contexto se recalculaba desde cero y las dependencias largas se diluían.

### 1.2 La Neurona Recurrente

Una **RNN** procesa la secuencia un elemento a la vez y **comparte los mismos pesos** en cada paso, conservando un **estado oculto** $h_t$ que resume el pasado:

$$h_t = \tanh\!\left(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b_h\right), \qquad \hat{y}_t = g\!\left(W_{hy}\,h_t + b_y\right)$$

| Arquitectura | Estado | Compuertas | Parámetros (aprox.) |
|---|---|---|---|
| **Simple RNN** | $h_t$ | 1 (implícita) | $d^2 + d\,k + d$ |
| **LSTM** | $h_t,\; C_t$ | Olvido, Entrada, Salida (3) | $4\,(d^2 + d\,k + d)$ |
| **GRU** | $h_t$ | Reset, Update (2) | $3\,(d^2 + d\,k + d)$ |

donde $d$ es la dimensión oculta y $k$ la dimensión de entrada.

### 1.3 El Tensor Tridimensional `[samples, time_steps, features]`

Este es **el cambio más importante** respecto al Lab 1. Keras exige que las capas recurrentes reciban un tensor **3D**:

| Dimensión | Símbolo | Significado | En este laboratorio |
|---|---|---|---|
| `samples` | $N$ | número de ventanas | 2 232 |
| `time_steps` | $T$ | longitud de la secuencia (retardos) | 24 |
| `features` | $d$ | variables medidas por instante | 1 (univariada) |

Cada muestra es una **matriz** $X^{(i)} \in \mathbb{R}^{T \times d}$:

$$X^{(i)} =
\begin{bmatrix}
x_{k-T,\,1} & \cdots & x_{k-T,\,d} \\
\vdots      &        & \vdots      \\
x_{k-1,\,1} & \cdots & x_{k-1,\,d}
\end{bmatrix}$$

> 💡 Si añadiéramos variables exógenas (temperatura, presión, humedad) tendríamos `features = 3`. La capa recurrente itera sobre `time_steps` y en cada paso lee el vector de `features`.

### 1.4 LSTM y GRU: compuertas que controlan la memoria

**LSTM** protege la información con una *autopista* de estado celular $C_t$ y tres compuertas:

$$f_t = \sigma\!\left(W_f[h_{t-1},x_t]+b_f\right),\quad
i_t = \sigma\!\left(W_i[h_{t-1},x_t]+b_i\right),\quad
o_t = \sigma\!\left(W_o[h_{t-1},x_t]+b_o\right)$$

$$C_t = f_t \odot C_{t-1} + i_t \odot \tanh\!\left(W_c[h_{t-1},x_t]+b_c\right), \qquad h_t = o_t \odot \tanh(C_t)$$

**GRU** simplifica el diseño fusionando estados y usando dos compuertas:

$$z_t = \sigma\!\left(W_z[h_{t-1},x_t]\right),\quad
r_t = \sigma\!\left(W_r[h_{t-1},x_t]\right)$$

$$h_t = (1 - z_t)\odot h_{t-1} + z_t \odot \tilde{h}_t$$

### 1.5 El Talón de Aquiles: Desvanecimiento del Gradiente

En *Backpropagation Through Time* el gradiente se multiplica por la matriz de pesos recurrentes en **cada paso**:

$$\frac{\partial \mathcal{L}_T}{\partial h_1} = \frac{\partial \mathcal{L}_T}{\partial h_T}\prod_{j=2}^{T}\frac{\partial h_j}{\partial h_{j-1}}$$

Si $\lVert W_{hh}\rVert < 1$ o la derivada de $\tanh$ es pequeña, el producto decae **exponencialmente**: la red "olvida" los primeros pasos. Lo demostraremos en la **Sección 8** con el *adding problem*.

---
## 2. ⚙️ Configuración Inicial

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tensorflow as tf
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import GRU, LSTM, Dense, Input, SimpleRNN
from tensorflow.keras.models import Model, Sequential

warnings.filterwarnings("ignore")

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
# Tensorflow desde la version 2.10 requiere cuda en windows WSL Windows Subsystem for Linux, para poder usar la GPU.
print(f"Keras version:      {tf.keras.__version__}")
print(f"GPU disponible:     {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 3. 📥 Carga del Dataset

Reutilizamos **Monthly Sunspots** (manchas solares mensuales, 1749–1983). El análisis espectral y la descomposición estacional ya se estudiaron en el Lab 1; aquí mostramos solo un resumen visual para recordar los **ciclos de Schwabe** (~11 años).

In [ ]:
URL_SUNSPOTS = (
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-sunspots.csv"
)

df = pd.read_csv(URL_SUNSPOTS)
df.columns = ["Fecha", "Sunspots"]
df["Fecha"] = pd.to_datetime(df["Fecha"])
df.set_index("Fecha", inplace=True)

# Serie como matriz columna (n, 1) de tipo float32 para Keras
values = df["Sunspots"].values.reshape(-1, 1).astype("float32")

print(f"Observaciones: {len(df)} meses")
print(f"Período: {df.index.min():%Y-%m} → {df.index.max():%Y-%m}")
df.head()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["Sunspots"],
        mode="lines",
        line=dict(color="#94a3b8", width=0.8),
        name="Serie mensual",
    )
)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["Sunspots"].rolling(12).mean(),
        mode="lines",
        line=dict(color="#f59e0b", width=2.5),
        name="Media móvil (12 meses)",
    )
)

fig.update_layout(
    title="Actividad Solar (1749–1983): Ciclos de Schwabe (~11 años)",
    template="plotly_white",
    height=420,
    hovermode="x unified",
    xaxis=dict(title="Año", gridcolor="#e2e8f0"),
    yaxis=dict(title="Número de manchas solares", gridcolor="#e2e8f0", zeroline=False),
    legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center"),
)
fig.show()

---
## 4. ✂️ Partición Temporal y Normalización

### ⚠️ Sin muestreo aleatorio en series de tiempo

Particionamos **cronológicamente** (80 % entrenamiento / 20 % prueba) para evitar *data leakage*, y ajustamos el escalador **solo** con el conjunto de entrenamiento:

$$x_{\text{scaled}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

In [ ]:
TRAIN_FRACTION = 0.8
train_size = int(len(values) * TRAIN_FRACTION)
train_raw, test_raw = values[:train_size], values[train_size:]

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_raw).astype("float32")
test_scaled = scaler.transform(test_raw).astype("float32")

print(f"Entrenamiento: {len(train_raw)} meses | Prueba: {len(test_raw)} meses")
print(f"Corte temporal: {df.index[train_size]:%Y-%m}")
print(
    f"Parámetros del escalador (solo train): μ = {scaler.mean_[0]:.2f} | σ = {scaler.scale_[0]:.2f}"
)

---
## 5. 🧱 Formulación de Secuencias: el Tensor 3D

Convertimos la serie en **ventanas deslizantes** y las apilamos en un tensor 3D:

$$\underbrace{\big[T_{k-T}, \dots, T_{k-1}\big]}_{\text{ventana}} \;\Longrightarrow\; X^{(i)} \in \mathbb{R}^{T \times d}, \qquad
\mathbf{X} \in \mathbb{R}^{N \times T \times d}$$

Con `features = 1` (univariada), cada ventana es una matriz de `24 × 1`. La RNN "leerá" esa columna paso a paso.

> 🧪 **Experimenta:** cambia `WINDOW_SIZE` (prueba 12, 48 o 132 meses) y vuelve a ejecutar el notebook para ver cómo cambia la memoria disponible.

In [ ]:
WINDOW_SIZE = 24


def build_sequences(series, window_size):
    """Convierte la serie en un tensor 3D [samples, time_steps, features]."""
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i : i + window_size])  # (time_steps, features)
        y.append(series[i + window_size])  # (1,)
    return np.asarray(X, dtype="float32"), np.asarray(y, dtype="float32").reshape(-1, 1)


X_train, y_train = build_sequences(train_scaled, WINDOW_SIZE)
X_test, y_test = build_sequences(test_scaled, WINDOW_SIZE)

# Vista plana (2D) que usaba la MLP del Laboratorio 1
X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

print("Tensor 3D [samples, time_steps, features]:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}   y_test: {y_test.shape}")
print(f"\nVista plana para la MLP (Lab 1): X_train_flat: {X_train_flat.shape}")

In [ ]:
print("Primera ventana X_train[0] — shape (time_steps, features):")
print(X_train[0].T)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "40 ventanas apiladas (samples × time_steps)",
        "Detalle: muestra 0",
    ],
    column_widths=[0.72, 0.28],
)

fig.add_trace(
    go.Heatmap(
        z=X_train[:40, :, 0].T,
        colorscale="Viridis",
        colorbar=dict(title="valor z", x=1.02),
        hovertemplate="sample %{x}<br>paso %{y}<br>z = %{z:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Heatmap(
        z=X_train[0].T,
        colorscale="Viridis",
        showscale=False,
        hovertemplate="feature %{x}<br>paso %{y}<br>z = %{z:.2f}<extra></extra>",
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="sample (ventana)", gridcolor="#e2e8f0", row=1, col=1)
fig.update_yaxes(title_text="time_steps", gridcolor="#e2e8f0", row=1, col=1)
fig.update_xaxes(title_text="features", row=1, col=2)
fig.update_yaxes(title_text="time_steps", row=1, col=2)
fig.update_layout(
    title="Representación de la secuencia como tensor 3D",
    template="plotly_white",
    height=430,
    margin=dict(t=80, r=110),
)
fig.show()

---
## 6. 🧠 Modelo LSTM: Arquitectura y Entrenamiento

### 6.1 De la ecuación a Keras

La capa `LSTM` de Keras implementa internamente las compuertas de la Sección 1.4. Con `units = 32` y una entrada de 24 pasos:

| Capa | Tipo | Forma de salida | Parámetros |
|---|---|---|---|
| Entrada | `Input` | `(None, 24, 1)` | 0 |
| Recurrente | `LSTM(32)` | `(None, 32)` | $4\,(32\cdot32 + 32\cdot1 + 32) = 4\,352$ |
| Salida | `Dense(1)` | `(None, 1)` | 33 |
| **Total** | | | **4 385** |

### 6.2 Entrenamiento con detención temprana

Usamos el optimizador **Adam**, la pérdida **MSE** y `EarlyStopping` para restaurar los mejores pesos y evitar el sobreajuste.

In [ ]:
UNITS = 32

lstm_model = Sequential(
    [
        Input(shape=(WINDOW_SIZE, 1), name="secuencia_entrada"),
        LSTM(UNITS, name="lstm"),
        Dense(1, activation="linear", name="salida"),
    ],
    name="LSTM_pronostico",
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.003), loss="mse", metrics=["mae"]
)
lstm_model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True, verbose=1
)

start = time.perf_counter()
lstm_history = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)
lstm_time = time.perf_counter() - start

print(f"\nTiempo de entrenamiento: {lstm_time:.1f} s")
print(f"Épocas efectivas: {len(lstm_history.history['loss'])}")
print(f"Mejor val_MSE: {min(lstm_history.history['val_loss']):.4f}")

In [ ]:
def plot_learning_curves(history, title):
    """Curvas de pérdida (MSE) y error absoluto (MAE) en train y validación."""
    epochs = range(1, len(history.history["loss"]) + 1)
    fig = make_subplots(
        rows=1, cols=2, subplot_titles=["Función de costo (MSE)", "MAE"]
    )
    for col, metric in enumerate(["loss", "mae"], start=1):
        fig.add_trace(
            go.Scatter(
                x=list(epochs),
                y=history.history[metric],
                mode="lines",
                name="Entrenamiento",
                line=dict(color="#4f46e5", width=2),
                showlegend=(col == 1),
                legendgroup="train",
            ),
            row=1,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=list(epochs),
                y=history.history[f"val_{metric}"],
                mode="lines",
                name="Validación",
                line=dict(color="#ef4444", width=2),
                showlegend=(col == 1),
                legendgroup="val",
            ),
            row=1,
            col=col,
        )
    fig.update_xaxes(title_text="Época", gridcolor="#e2e8f0")
    fig.update_yaxes(gridcolor="#e2e8f0")
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=400,
        legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"),
    )
    fig.show()


plot_learning_curves(lstm_history, "Entrenamiento del Modelo LSTM")

In [ ]:
# Guardamos el resultado del LSTM para el benchmarking
results = {
    "LSTM": {
        "kind": "lstm",
        "model": lstm_model,
        "history": lstm_history,
        "time": lstm_time,
        "params": lstm_model.count_params(),
        "epochs": len(lstm_history.history["loss"]),
        "best_val": min(lstm_history.history["val_loss"]),
    }
}
print("Modelo LSTM registrado.")

---
## 7. 🏁 Benchmarking: MLP vs. Simple RNN vs. LSTM vs. GRU

Comparamos las cuatro arquitecturas bajo las **mismas condiciones**: mismo `window_size`, misma dimensión oculta (`units = 32`), mismo optimizador y criterio de parada. Así aislamos el efecto de la **arquitectura**.

- **MLP** (Lab 1): recibe la ventana plana `(24,)` — sin memoria explícita.
- **Simple RNN / LSTM / GRU**: reciben la secuencia `(24, 1)`.

Registramos tres dimensiones de costo: **parámetros**, **tiempo de entrenamiento** y **MSE de validación**.

> 🧪 **Experimenta:** cambia `UNITS` (por ejemplo 16 o 64) y observa el compromiso entre capacidad, tiempo y sobreajuste.

In [ ]:
COLORS = {
    "MLP": "#64748b",
    "Simple RNN": "#ef4444",
    "LSTM": "#4f46e5",
    "GRU": "#10b981",
}


def build_sunspot_model(kind, units=UNITS, window=WINDOW_SIZE):
    """Construye el modelo según la arquitectura solicitada."""
    if kind == "mlp":
        model = Sequential(
            [
                Input(shape=(window,), name="ventana_plana"),
                Dense(units, activation="relu", name="oculta"),
                Dense(1, activation="linear", name="salida"),
            ],
            name="MLP_AR",
        )
    else:
        layer_cls = {"rnn": SimpleRNN, "lstm": LSTM, "gru": GRU}[kind]
        model = Sequential(
            [
                Input(shape=(window, 1), name="secuencia_entrada"),
                layer_cls(units, name="recurrente"),
                Dense(1, activation="linear", name="salida"),
            ],
            name=f"{kind.upper()}_AR",
        )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
        loss="mse",
        metrics=["mae"],
    )
    return model


def train_sunspot_model(kind, epochs=150, batch_size=32, patience=20):
    """Entrena una arquitectura y devuelve (modelo, historia, tiempo)."""
    X_tr = X_train_flat if kind == "mlp" else X_train
    X_va = X_test_flat if kind == "mlp" else X_test
    model = build_sunspot_model(kind)
    early = EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0
    )
    start = time.perf_counter()
    history = model.fit(
        X_tr,
        y_train,
        validation_data=(X_va, y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early],
        verbose=0,
    )
    return model, history, time.perf_counter() - start


for label, kind in {"MLP": "mlp", "Simple RNN": "rnn", "GRU": "gru"}.items():
    model, history, elapsed = train_sunspot_model(kind)
    results[label] = {
        "kind": kind,
        "model": model,
        "history": history,
        "time": elapsed,
        "params": model.count_params(),
        "epochs": len(history.history["loss"]),
        "best_val": min(history.history["val_loss"]),
    }
    print(
        f"{label:<11} parámetros={model.count_params():>6} | "
        f"épocas={len(history.history['loss']):>3} | "
        f"tiempo={elapsed:6.1f}s | val_MSE={min(history.history['val_loss']):.4f}"
    )

In [ ]:
fig = go.Figure()
for label, r in results.items():
    fig.add_trace(
        go.Scatter(
            y=r["history"].history["val_loss"],
            mode="lines",
            name=label,
            line=dict(color=COLORS[label], width=2.2),
        )
    )

fig.update_layout(
    title="Convergencia en validación (MSE por época)",
    xaxis=dict(title="Época", gridcolor="#e2e8f0"),
    yaxis=dict(title="val_MSE (escala log)", gridcolor="#e2e8f0", type="log"),
    template="plotly_white",
    height=440,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"),
)
fig.show()

In [ ]:
labels = list(results.keys())
params = [results[l]["params"] for l in labels]
times = [results[l]["time"] for l in labels]
best_val = [results[l]["best_val"] for l in labels]
colors = [COLORS[l] for l in labels]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Parámetros entrenables",
        "Tiempo de entrenamiento (s)",
        "Mejor MSE de validación",
    ],
)

for col, (vals, fmt) in enumerate(
    [(params, "{:.0f}"), (times, "{:.1f}"), (best_val, "{:.4f}")], start=1
):
    fig.add_trace(
        go.Bar(
            x=labels,
            y=vals,
            marker_color=colors,
            text=[fmt.format(v) for v in vals],
            textposition="outside",
            showlegend=False,
            hovertemplate="%{x}: %{y}<extra></extra>",
        ),
        row=1,
        col=col,
    )

fig.update_yaxes(gridcolor="#e2e8f0")
fig.update_layout(
    title="Costo vs. capacidad vs. rendimiento",
    template="plotly_white",
    height=450,
    margin=dict(t=100),
)
fig.show()

---
## 8. 🎯 Predicción a Un Paso (*One-Step-Ahead*)

Evaluamos cada modelo sobre el conjunto de prueba. Recordemos que las predicciones se generan en escala normalizada: hay que **invertir el escalador** para interpretarlas en número de manchas solares.

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{N}\sum (y_i - \hat{y}_i)^2}$ | Error en las unidades de la variable (manchas) |
| **MAE** | $\frac{1}{N}\sum \lvert y_i - \hat{y}_i\rvert$ | Error absoluto promedio (robusto a *outliers*) |

In [ ]:
y_test_real = scaler.inverse_transform(y_test)

one_step = {}
for label, r in results.items():
    X_in = X_test_flat if r["kind"] == "mlp" else X_test
    pred_real = scaler.inverse_transform(r["model"].predict(X_in, verbose=0))
    rmse = float(np.sqrt(np.mean((y_test_real - pred_real) ** 2)))
    mae = float(np.mean(np.abs(y_test_real - pred_real)))
    one_step[label] = {"pred": pred_real, "rmse": rmse, "mae": mae}
    print(f"{label:<11} RMSE={rmse:6.2f} | MAE={mae:6.2f} manchas solares")

In [ ]:
ZOOM = 120  # ~1 ciclo solar completo (~10 años)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        y=y_test_real[:ZOOM, 0],
        mode="lines",
        name="Observaciones reales",
        line=dict(color="#0f172a", width=2.6),
        hovertemplate="Mes %{x}<br>Real: %{y:.1f}<extra></extra>",
    )
)

for label, r in one_step.items():
    fig.add_trace(
        go.Scatter(
            y=r["pred"][:ZOOM, 0],
            mode="lines",
            name=f"{label} (RMSE={r['rmse']:.1f})",
            line=dict(color=COLORS[label], width=1.7, dash="dash"),
            hovertemplate=f"{label}: %{{y:.1f}}<extra></extra>",
        )
    )

fig.update_layout(
    title=f"Predicción a un paso por arquitectura — primeros {ZOOM} meses de prueba",
    template="plotly_white",
    height=480,
    hovermode="x unified",
    xaxis=dict(title="Meses consecutivos de prueba", gridcolor="#e2e8f0"),
    yaxis=dict(title="Manchas solares", gridcolor="#e2e8f0", zeroline=False),
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"),
)
fig.show()

---
## 9. 🔁 Inferencia Recurrente y Proyecciones Futuras

### 9.1 El estado de memoria, paso a paso

Las capas recurrentes exponen su estado interno. Reconstruimos el LSTM ya entrenado como un modelo de **un solo paso** y lo alimentamos manualmente, observando cómo evoluciona el estado oculto $h_t$ y demostrando que coincide con la predicción del modelo completo.

> ✅ El resultado del "paso a paso" debe ser **idéntico** al del modelo que procesa la secuencia completa: es la misma recurrencia, solo que ejecutada explícitamente.

### 9.2 Pronóstico recursivo multi-paso

En producción no conocemos el futuro, así que realimentamos la ventana con **las propias predicciones**:

1. Predecir $\hat{T}_k = f_\theta([T_{k-T}, \dots, T_{k-1}])$
2. Desplazar la ventana: eliminar el valor más antiguo, añadir $\hat{T}_k$
3. Repetir hasta completar el horizonte $H$

El error se **acumula** con el horizonte. Compararemos la estabilidad de las arquitecturas.

In [ ]:
# --- Reconstrucción del LSTM como modelo de un paso ---
lstm_layer = lstm_model.layers[0]
dense_layer = lstm_model.layers[1]
recurrent_units = lstm_layer.units

x_in = Input(shape=(1, 1), name="x_t")
h_in = Input(shape=(recurrent_units,), name="h_prev")
c_in = Input(shape=(recurrent_units,), name="c_prev")

lstm_step = LSTM(recurrent_units, return_state=True, name="lstm_step")
h_t, new_h, new_c = lstm_step(x_in, initial_state=[h_in, c_in])
y_t = Dense(1, name="salida_step")(new_h)
step_model = Model([x_in, h_in, c_in], [y_t, new_h, new_c])

# Transferimos los pesos aprendidos
lstm_step.set_weights(lstm_layer.get_weights())
step_model.get_layer("salida_step").set_weights(dense_layer.get_weights())

# --- Ejecución manual sobre una ventana de prueba ---
window = tf.constant(X_test[0].reshape(1, WINDOW_SIZE, 1))
pred_full = float(lstm_model(window, training=False).numpy()[0, 0])

h = tf.zeros((1, recurrent_units))
c = tf.zeros((1, recurrent_units))
norms_h, preds_step = [], []
for t in range(WINDOW_SIZE):
    x_t = tf.constant(X_test[0, t : t + 1].reshape(1, 1, 1))
    y_t_val, h, c = step_model([x_t, h, c], training=False)
    norms_h.append(float(tf.norm(h)))
    preds_step.append(float(y_t_val[0, 0]))

print(f"Modelo completo   : {pred_full:.6f}")
print(f"Paso a paso manual: {preds_step[-1]:.6f}")
print(f"Diferencia        : {abs(pred_full - preds_step[-1]):.2e}")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=list(range(1, WINDOW_SIZE + 1)),
        y=norms_h,
        mode="lines+markers",
        line=dict(color="#4f46e5", width=2),
        marker=dict(size=6, color="#4f46e5"),
        name="‖h_t‖",
    )
)
fig.update_layout(
    title="Evolución del estado oculto h_t mientras el LSTM lee la secuencia",
    xaxis=dict(title="Paso temporal t", gridcolor="#e2e8f0", dtick=2),
    yaxis=dict(title="Norma del estado oculto ‖h_t‖", gridcolor="#e2e8f0"),
    template="plotly_white",
    height=400,
)
fig.show()

In [ ]:
def forecast_from_window(model, window, horizon):
    """Pronóstico recursivo desde una única ventana (time_steps, features)."""
    window = np.asarray(window, dtype="float32").reshape(WINDOW_SIZE, 1).copy()
    preds = []
    for _ in range(horizon):
        tensor = tf.convert_to_tensor(
            window.reshape(1, WINDOW_SIZE, 1), dtype=tf.float32
        )
        next_val = float(model(tensor, training=False).numpy()[0, 0])
        preds.append(next_val)
        window = np.roll(window, -1, axis=0)
        window[-1, 0] = next_val
    return np.asarray(preds).reshape(-1, 1)


def recursive_forecast(model, X_start, horizon):
    """Pronóstico recursivo por bloques sobre un conjunto de ventanas."""
    total = len(X_start) - horizon + 1
    preds = []
    for i in range(0, total, horizon):
        preds.extend(forecast_from_window(model, X_start[i], horizon).flatten())
    return np.asarray(preds).reshape(-1, 1)


HORIZONS = [1, 3, 6, 12]
horizon_rmse = {label: [] for label in ["Simple RNN", "LSTM", "GRU"]}

for label in horizon_rmse:
    for hz in HORIZONS:
        pred_real = scaler.inverse_transform(
            recursive_forecast(results[label]["model"], X_test, horizon=hz)
        )
        rmse = float(np.sqrt(np.mean((y_test_real[: len(pred_real)] - pred_real) ** 2)))
        horizon_rmse[label].append(rmse)
    print(
        f"{label:<11}",
        "  ".join(f"H{h}={v:6.2f}" for h, v in zip(HORIZONS, horizon_rmse[label])),
    )

In [ ]:
fig = go.Figure()
for label, rmses in horizon_rmse.items():
    fig.add_trace(
        go.Scatter(
            x=HORIZONS,
            y=rmses,
            mode="lines+markers",
            name=label,
            line=dict(color=COLORS[label], width=2.4),
            marker=dict(size=9),
            hovertemplate=f"{label}<br>Horizonte %{{x}}<br>RMSE: %{{y:.2f}}<extra></extra>",
        )
    )

fig.update_layout(
    title="Estabilidad del pronóstico recursivo: RMSE vs. horizonte",
    xaxis=dict(title="Horizonte recursivo H (meses)", gridcolor="#e2e8f0", dtick=1),
    yaxis=dict(title="RMSE (manchas solares)", gridcolor="#e2e8f0"),
    template="plotly_white",
    height=430,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"),
)
fig.show()

In [ ]:
# --- Proyección futura: 24 meses después de 1983 con el mejor modelo LSTM ---
FUTURE_H = 24
full_scaled = scaler.transform(values).astype("float32")
last_window = full_scaled[-WINDOW_SIZE:]

future_real = scaler.inverse_transform(
    forecast_from_window(lstm_model, last_window, horizon=FUTURE_H)
)
future_dates = pd.date_range(df.index[-1], periods=FUTURE_H + 1, freq="MS")[1:]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df.index[-180:],
        y=df["Sunspots"].values[-180:],
        mode="lines",
        name="Histórico",
        line=dict(color="#0f172a", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=future_dates,
        y=future_real[:, 0],
        mode="lines+markers",
        name="Proyección LSTM (24 meses)",
        line=dict(color="#4f46e5", width=2.4, dash="dash"),
        marker=dict(size=6),
    )
)
fig.update_layout(
    title="Proyección del ciclo solar con LSTM recursivo (más allá de 1983)",
    template="plotly_white",
    height=450,
    hovermode="x unified",
    xaxis=dict(title="Año", gridcolor="#e2e8f0"),
    yaxis=dict(title="Manchas solares", gridcolor="#e2e8f0", zeroline=False),
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"),
)
fig.show()

---
## 10. 🧠 Memoria a Largo Plazo: el *Adding Problem*

### La tarea

Para medir la **retención de memoria** construimos una tarea sintética clásica (Hochreiter & Schmidhuber, 1997). Cada secuencia tiene **dos canales**:

- **Canal 0:** valores aleatorios $v_t \sim \mathcal{U}(0,1)$.
- **Canal 1:** un marcador `1.0` en **dos posiciones aleatorias** y `0.0` en el resto.

El objetivo es **sumar los valores del canal 0 en las posiciones marcadas**:

$$y = v_{i_1} + v_{i_2}, \qquad \text{con } i_1, i_2 \text{ marcados}$$

La red debe recordar *dónde* estaban los marcadores a lo largo de toda la secuencia. Es el escenario ideal para exponer el **desvanecimiento del gradiente**: una Simple RNN no puede propagar la información desde los primeros pasos, mientras que LSTM y GRU mantienen una *autopista* de memoria.

> 🎯 **Hipótesis:** a mayor longitud de secuencia, la Simple RNN degrada hasta el *baseline* (predecir la media), mientras que LSTM y GRU se mantienen.

In [ ]:
def build_adding_problem(n_samples, seq_len, seed=0):
    """Genera el adding problem: sumar los valores de las dos posiciones marcadas."""
    rng = np.random.default_rng(seed)
    X = np.zeros((n_samples, seq_len, 2), dtype="float32")
    X[:, :, 0] = rng.random((n_samples, seq_len))

    i1 = rng.integers(0, seq_len, n_samples)
    i2 = rng.integers(0, seq_len, n_samples)
    same = i1 == i2
    i2[same] = (i2[same] + 1) % seq_len  # garantizar posiciones distintas

    rows = np.arange(n_samples)
    X[rows, i1, 1] = 1.0
    X[rows, i2, 1] = 1.0
    y = (X[rows, i1, 0] + X[rows, i2, 0]).reshape(-1, 1)
    return X, y


X_demo, y_demo = build_adding_problem(3, 12, seed=0)
print("Forma de X:", X_demo.shape, "| Forma de y:", y_demo.shape)
print("Marcadores por muestra:", X_demo[:, :, 1].sum(axis=1))
print("Etiquetas (suma de los valores marcados):", y_demo.flatten().round(3))

In [ ]:
SEQ_LENGTHS = [20, 50, 100]
memory_results = {"Simple RNN": [], "LSTM": [], "GRU": []}
baselines = []


def build_memory_model(kind, units=32):
    layer_cls = {"rnn": SimpleRNN, "lstm": LSTM, "gru": GRU}[kind]
    model = Sequential(
        [
            Input(shape=(None, 2), name="secuencia_entrada"),
            layer_cls(units, name="recurrente"),
            Dense(1, activation="linear", name="salida"),
        ]
    )
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003), loss="mse")
    return model


for seq_len in SEQ_LENGTHS:
    X_tr, y_tr = build_adding_problem(2000, seq_len, seed=1)
    X_va, y_va = build_adding_problem(500, seq_len, seed=2)
    baselines.append(float(np.mean((y_va - y_va.mean()) ** 2)))

    for label, kind in {"Simple RNN": "rnn", "LSTM": "lstm", "GRU": "gru"}.items():
        model = build_memory_model(kind)
        early = EarlyStopping(
            monitor="val_loss", patience=20, restore_best_weights=True, verbose=0
        )
        history = model.fit(
            X_tr,
            y_tr,
            validation_data=(X_va, y_va),
            epochs=120,
            batch_size=32,
            callbacks=[early],
            verbose=0,
        )
        memory_results[label].append(min(history.history["val_loss"]))

    print(
        f"T={seq_len:>3} | "
        + " | ".join(
            f"{label}={memory_results[label][-1]:.4f}" for label in memory_results
        )
        + f" | baseline={baselines[-1]:.4f}"
    )

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=SEQ_LENGTHS,
        y=baselines,
        mode="lines",
        name="Baseline (predecir la media)",
        line=dict(color="#94a3b8", width=2, dash="dot"),
    )
)
for label, vals in memory_results.items():
    fig.add_trace(
        go.Scatter(
            x=SEQ_LENGTHS,
            y=vals,
            mode="lines+markers",
            name=label,
            line=dict(color=COLORS[label], width=2.6),
            marker=dict(size=10),
            hovertemplate=f"{label}<br>Longitud %{{x}}<br>MSE: %{{y:.4f}}<extra></extra>",
        )
    )

fig.update_layout(
    title="Memoria a largo plazo: error vs. longitud de la secuencia",
    xaxis=dict(
        title="Longitud de la secuencia (time_steps)", gridcolor="#e2e8f0", type="log"
    ),
    yaxis=dict(title="MSE de validación (escala log)", gridcolor="#e2e8f0", type="log"),
    template="plotly_white",
    height=460,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"),
)
fig.show()

---
## 11. 🎓 Conclusiones

1. **Tensor 3D `[samples, time_steps, features]`:** es el contrato de entrada de toda capa recurrente en Keras. La misma ventana que en el Lab 1 era un vector plano de 24 valores, aquí se reinterpreta como una secuencia de 24 pasos con 1 variable.

2. **Memoria explícita:** a diferencia del MLP, la RNN mantiene un estado $h_t$ que se actualiza paso a paso. La "inferencia recurrente" paso a paso reproduce **exactamente** el resultado del modelo completo.

3. **El costo de las compuertas:** LSTM ($4d^2$) y GRU ($3d^2$) tienen más parámetros y tardan más por época que la Simple RNN ($d^2$), pero en el ciclo solar su rendimiento es igual o mejor.

4. **Error acumulado:** el RMSE del pronóstico recursivo crece con el horizonte $H$, porque la ventana se realimenta con predicciones que contienen error. La predicción a un paso es siempre un límite inferior optimista.

5. **Desvanecimiento del gradiente:** en el *adding problem* la Simple RNN **colapsa al baseline** en secuencias largas ($T=100$), mientras que LSTM y GRU resuelven la tarea con MSE ≈ 0. Las compuertas y la autopista del estado celular preservan el gradiente a lo largo del tiempo.

### 🚀 Próximos pasos

- Probar **redes bidireccionales** (`Bidirectional(LSTM(...))`) cuando toda la secuencia está disponible.
- Apilar varias capas recurrentes (`return_sequences=True` entre capas).
- Usar **features exógenas** (actividad magnética, series multivariadas).
- Comparar con **Transformers**, cuyos mecanismos de atención atacan directamente el problema de las dependencias largas.

---

**📚 Referencias:**

[1] Hochreiter, S. & Schmidhuber, J. (1997). *Long Short-Term Memory*. Neural Computation, 9(8), 1735–1780.  
[2] Cho, K. et al. (2014). *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*. EMNLP.  
[3] Bengio, Y., Simard, P. & Frasconi, P. (1994). *Learning Long-Term Dependencies with Gradient Descent is Difficult*. IEEE Transactions on Neural Networks.  
[4] Chollet, F. (2021). *Deep Learning with Python*. 2nd ed. Manning Publications.  
[5] [Brownlee, J. — Monthly Sunspots Dataset](https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-sunspots.csv)